# 04 — One-stage anchor-based: RetinaNet

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/vod").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q faster-coco-eval

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vod import CLASSES
from vod.data import find_data_root, load_split
from vod.protocol import PROTOCOL
from vod.submission import write_submission
from vod.tv import (RETINA_SIZES, anchor_match_stats, comparison_tables, completed_runs, load_run_model,
                    predict_test, run_all)

DATA = find_data_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")

## Data (same protocol as every detector)

In [ ]:
train_frames, train_targets = load_split(DATA / "train", step=PROTOCOL["train_step"])
val_frames, val_targets = load_split(DATA / "val", step=1)
VAL_GT = DATA / "val" / "annotations_coco.json"
print(len(train_frames), "train frames,", len(val_frames), "val frames")

## Anchors grounded in the training boxes

In [ ]:
# aspect ratios (h/w) at the 1/6, 1/2 and 5/6 quantiles of the training boxes
wh = np.concatenate([train_targets[i][0][:, 2:] - train_targets[i][0][:, :2] for i in train_frames["id"]])
log_ratio = np.log(wh[:, 1] / wh[:, 0])
DATA_RATIOS = tuple(np.round(np.exp(np.quantile(log_ratio, [1 / 6, 1 / 2, 5 / 6])), 2).tolist())
print("data aspect ratios h/w:", DATA_RATIOS)

ANCHOR_CANDIDATES = {
    "default 32-512, ratios 0.5/1/2": (RETINA_SIZES, (0.5, 1.0, 2.0)),
    "16-256, ratios 0.5/1/2": ((16, 32, 64, 128, 256), (0.5, 1.0, 2.0)),
    f"16-256, ratios {DATA_RATIOS}": ((16, 32, 64, 128, 256), DATA_RATIOS),
    f"12-192, ratios {DATA_RATIOS}": ((12, 24, 48, 96, 192), DATA_RATIOS),
}
ANCHOR_ANALYSIS = True  # set False to skip (a few CPU minutes) when only re-running evaluation or test
coverage = {}
for name, (sizes, ratios) in (ANCHOR_CANDIDATES.items() if ANCHOR_ANALYSIS else []):
    for fg in [0.5, 0.4]:
        t = anchor_match_stats(train_frames, train_targets, sizes, ratios, PROTOCOL["imgsz"], fg)
        coverage[(name, fg)] = t
        print(f"\n{name}, fg IoU {fg}: {t.attrs['anchors_per_image']} anchors/image, "
              f"negatives per positive anchor ~{t.attrs['negative_to_positive']}")
        display(t)

In [ ]:
if coverage:
    summary = pd.DataFrame({k: v["no_positive_anchor"] for k, v in coverage.items()}).T
    summary.index.names = ["anchors", "fg_iou"]
    summary.columns = [f"no positive, {c}" for c in summary.columns]
    display(summary)

## Experiments
Each experiment changes one thing relative to the previous one. All of `RUN` are trained one after
another (~1.5 h each on a T4); every finished run is saved right away, so a crash keeps earlier runs.

In [ ]:
# data ratios (0.56, 1.06, 1.96) cover the boxes as well as the default (0.5, 1, 2), so only the scale changes
BASE = dict(arch="retinanet", lr=0.0025)
SMALL = dict(anchor_sizes=(16, 32, 64, 128, 256))
EXPERIMENTS = {
    "retina_default": BASE,
    "retina_anchors": {**BASE, **SMALL},
    "retina_anchors_iou": {**BASE, **SMALL, "fg_iou": 0.4, "bg_iou": 0.3},
    "retina_anchors_iou_focal": {**BASE, **SMALL, "fg_iou": 0.4, "bg_iou": 0.3, "focal_gamma": 1.5},
    # plain sigmoid cross-entropy: what happens without the fix for foreground/background imbalance
    "retina_no_focal": {**BASE, "focal_gamma": 0.0},
}
RUN = list(EXPERIMENTS)

## Train and evaluate (last epoch, every val frame)
Runs attached as input are not retrained; runs evaluated with older code (best epoch) are re-evaluated
from their last.pt so that every number comes from the same procedure.

In [ ]:
run_all(RUN, EXPERIMENTS, PROTOCOL, train_frames, train_targets, val_frames, val_targets, VAL_GT, WORK)

## Comparison with every finished run
Attach the Faster R-CNN outputs (03) as input to include the tuned two-stage detector.

In [ ]:
runs = completed_runs((WORK, Path("/kaggle/input")))
main, per_class, per_clip = comparison_tables(runs, CLASSES)
display(main)
display(per_class)
per_clip  # mAP of each val clip: is a difference consistent across scenes?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for n, m in runs.items():
    if m.get("history") is not None:
        ax.plot(m["history"]["epoch"], m["history"]["mAP"], marker="o", label=n)
ax.set(xlabel="epoch", ylabel="val mAP (every 10th frame)", title="Validation mAP during training")
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(WORK / "map_curves.png", dpi=110)

## Test predictions with the best RetinaNet run (by val mAP)

In [ ]:
BEST = main[main.index.isin(EXPERIMENTS)]["mAP"].idxmax()
cfg = {**PROTOCOL, **EXPERIMENTS[BEST]}
model = load_run_model(BEST, cfg, (WORK, Path("/kaggle/input")))
print("best:", BEST)
sub = write_submission(predict_test(model, cfg, DATA), DATA / "test_images.csv", WORK / f"submission_{BEST}.csv")
sub.head()